In [ ]:
#imports
import nibabel as nib
#import nilearn
import numpy as np
import os
import cv2

import cc3d
from matplotlib import pyplot as plt

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

In [ ]:
#from tqdm import tqdm
from skimage.io import imread, imshow
from skimage.transform import resize

In [ ]:
import tensorflow as tf

import keras.backend as K
from keras.models import Model, load_model
from keras.layers import Input, BatchNormalization, Activation, Dense, Dropout
from keras.layers.core import Lambda, RepeatVector, Reshape
from keras.layers.convolutional import Conv2D, Conv2DTranspose
from keras.layers.pooling import MaxPooling2D, GlobalMaxPool2D
from tensorflow.keras.layers import concatenate
#from keras.layers.merge import concatenate, add
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from keras.callbacks import CSVLogger
#from keras.preprocessing.image import ImageDataGenerator, array_to_img, img_to_array, load_img
from tensorflow.keras.models import *
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import *
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping, TensorBoard
from tensorflow.keras.layers.experimental import preprocessing

from tensorflow.keras.losses import binary_crossentropy

#from focal_loss import BinaryFocalLoss

In [ ]:
TRAIN_DATASET_PATH = 'C:\\Users\\Anita\\Datasets\\ISLES2022\\Training\\rawdata'
train_directories = [f.path for f in os.scandir(TRAIN_DATASET_PATH) if f.is_dir()]
#len(train_directories)
train_directories
train_ids = []
for i in range(len(train_directories)):
    train_ids.append(train_directories[i][51:69])
len(train_ids)

In [ ]:
TRAINMask_DATASET_PATH = 'C:\\Users\\Anita\\Datasets\\ISLES2022\\Training\\derivatives'
mask_directories = [f.path for f in os.scandir(TRAINMask_DATASET_PATH) if f.is_dir()]
mask_directories
len(mask_directories)

mask_ids = []
for i in range(len(mask_directories)):
    mask_ids.append(mask_directories[i][55:73])
len(mask_ids)

In [ ]:
from sklearn.model_selection import train_test_split
train_val_ids, test_ids,train_val_mask, test_mask = train_test_split(train_ids,mask_ids,test_size=0.20,random_state=42) 
train_ids,  val_ids, train_mask , val_mask = train_test_split(train_val_ids,train_val_mask,test_size=0.20,random_state=42)
len(train_ids)

In [ ]:
print('trainingId:',len(train_ids))
print('val_Id:',len(val_ids))
print('Test_Id:',len(test_ids))

In [ ]:
import cv2
import numpy as np

IMG_SIZE = 128


def Crop_or_pad_image(image, target_shape, coord_dict=None):
    height, width = image.shape[:2]
    target_height, target_width = target_shape

    if height > target_height or width > target_width:
        if coord_dict is not None and (height, width) in coord_dict:
            y1, y2, x1, x2 = coord_dict[(height, width)]
            image = image[y1:y2, x1:x2]
            return image   
    else:
        # Pad smaller images to target size
        pad_height = (target_height - height) // 2
        pad_width = (target_width - width) // 2

        pad_height_top = pad_height
        pad_height_bottom = target_height - height - pad_height_top
        pad_width_left = pad_width
        pad_width_right = target_width - width - pad_width_left

        padded_image = np.pad(image, ((pad_height_top, pad_height_bottom), 
                                      (pad_width_left, pad_width_right)), 
                              mode='constant', constant_values=0)
        return padded_image

# Example usage
def manual_crop(image, crop_coords):
    x_start, y_start, x_end, y_end = crop_coords
    image = image[x_start:y_start, x_end:y_end]
    return image

manual_crop_coords = (35, 163, 10, 138)
# Dictionary of coordinates for cropping based on image size
coord_dict = {
    (130, 130): (1, 129, 1, 129),
    (170, 170): (20, 148, 18, 146),
    (192, 192): (35, 163, 30, 158),
    (256, 256): (65, 193, 60, 188) #(35, 220, 25, 210)
    # Add more sizes and coordinates as needed
}

In [ ]:
class DataGenerator(tf.keras.utils.Sequence):
    def __init__(self, list_IDs, dim=(IMG_SIZE,IMG_SIZE), batch_size = 1, n_channels = 1, shuffle=False):
       
        self.dim = dim
        self.batch_size = batch_size
        self.list_IDs = list_IDs
        self.n_channels = n_channels
        self.shuffle = shuffle
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.list_IDs) / self.batch_size))

    def __getitem__(self, index):
        indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]

        Batch_ids = [self.list_IDs[k] for k in indexes]

        X, y = self.__data_generation(Batch_ids)

        return X, y

    def on_epoch_end(self):
        self.indexes = np.arange(len(self.list_IDs))
        if self.shuffle == True:
            np.random.shuffle(self.indexes)

    def __data_generation(self, Batch_ids):
        
        # Generate data
        for c, i in enumerate(Batch_ids):        
            case_path = os.path.join(TRAIN_DATASET_PATH, i)
            data_path = os.path.join(case_path + '/ses-0001', f'{i}_ses-0001_dwi.nii.gz');
            dwi = nib.load(data_path).get_fdata()
            dwi=scaler.fit_transform(dwi.reshape(-1, dwi.shape[-1])).reshape(dwi.shape)
            #adc=scaler.fit_transform(adc.reshape(-1, adc.shape[-1])).reshape(adc.shape)
            slices = dwi.shape[2]
            X = np.zeros((slices, IMG_SIZE,IMG_SIZE, 1))

            case_path2 = os.path.join(TRAINMask_DATASET_PATH, i)
            data_path_2 = os.path.join(case_path2 + '/ses-0001', f'{i}_ses-0001_msk.nii.gz');
            msk = nib.load(data_path_2).get_fdata()
            #msk=msk.astype(np.float32)
            msk_slices = msk.shape[2]
            y = np.zeros((msk_slices, IMG_SIZE,IMG_SIZE))
            #print(msk.dtype)  

            
            for j in range(slices):
                # Manually crop if mask_id is 238,184
                if i == "sub-strokecase0184" or i == "sub-strokecase0238":
                    X[j,:,:,0] = manual_crop(dwi[:,:,j+0], manual_crop_coords)
                    y[j] = manual_crop(msk[:,:,j+0], manual_crop_coords)
                else:
                    X[j,:,:,0] = Crop_or_pad_image(dwi[:,:,j+0],(IMG_SIZE,IMG_SIZE),coord_dict);
                    y[j] = Crop_or_pad_image(msk[:,:,j+0],(IMG_SIZE,IMG_SIZE),coord_dict);
                X=X.astype(np.float32)                
                # Apply thresholding to convert back to binary values
                threshold = 0.1  # Adjust this threshold as needed
                binary_mask = np.where(y >= threshold, 1, 0).astype(np.float32)
                

        return X, binary_mask
        
training_generator = DataGenerator(train_ids)
val_generator = DataGenerator(val_ids)
test_generator = DataGenerator(test_ids)

In [ ]:
test_pred=model.predict(test_generator)

In [ ]:
import tensorflow as tf

import numpy as np
from scipy.spatial.distance import cdist


def hausdorff_distance(y_true, y_pred, spacing=(1.0, 1.0), threshold=0.5):
    """
    Symmetric Hausdorff Distance (HD) for 2D binary segmentation masks.

    Parameters
    ----------
    y_true : numpy.ndarray
        Ground-truth binary mask, shape (H, W) or (H, W, 1).

    y_pred : numpy.ndarray
        Predicted probability map, shape (H, W) or (H, W, 1).

    spacing : tuple
        Physical pixel spacing in mm: (row_spacing, col_spacing).

    threshold : float
        Threshold used to convert predicted probabilities to binary mask.

    Returns
    -------
    hd : float
        Symmetric Hausdorff Distance in millimeters.
    """

    # Remove singleton channel dimension if present
    y_true = np.squeeze(y_true)
    y_pred = np.squeeze(y_pred)

    # Convert prediction probability to binary mask
    y_true = y_true > 0
    y_pred = y_pred > threshold

    # Get foreground coordinates
    true_coords = np.argwhere(y_true)
    pred_coords = np.argwhere(y_pred)

    # ---------------------------------------------------------
    # Case 1: Both masks are empty
    # ---------------------------------------------------------
    if len(true_coords) == 0 and len(pred_coords) == 0:
        return 0.0

    # ---------------------------------------------------------
    # Case 2: One mask is empty
    # ---------------------------------------------------------
    if len(true_coords) == 0 or len(pred_coords) == 0:
        return np.inf

    # ---------------------------------------------------------
    # Convert pixel coordinates to physical coordinates (mm)
    # ---------------------------------------------------------
    true_coords = true_coords.astype(np.float64)
    pred_coords = pred_coords.astype(np.float64)

    true_coords[:, 0] *= spacing[0]
    true_coords[:, 1] *= spacing[1]

    pred_coords[:, 0] *= spacing[0]
    pred_coords[:, 1] *= spacing[1]

    # ---------------------------------------------------------
    # Pairwise Euclidean distances in mm
    # ---------------------------------------------------------
    distances = cdist(true_coords, pred_coords, metric='euclidean')

    # GT -> Prediction
    directed_gt_to_pred = np.min(distances, axis=1)
    hd_gt_to_pred = np.max(directed_gt_to_pred)

    # Prediction -> GT
    directed_pred_to_gt = np.min(distances, axis=0)
    hd_pred_to_gt = np.max(directed_pred_to_gt)

    # Symmetric Hausdorff Distance
    hd = max(hd_gt_to_pred, hd_pred_to_gt)

    return float(hd)


# Function to compute the Dice coefficient
def dice_coeff(y_true,y_pred):
    # Ensure both tensors are of type float32
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    y_true_new = K.flatten(y_true)
    y_pred_new = K.flatten(y_pred)
    denominator = K.sum(y_true_new) + K.sum(y_pred_new)
    if denominator == 0.0:
        return 1.0
    numerator = K.sum(y_true_new * y_pred_new)
    return (2.0*numerator)/(denominator)

def iou(y_true,y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    y_true = K.flatten(y_true)
    y_pred = K.flatten(y_pred)
    intersec = K.sum(y_true * y_pred)
    union = K.sum(y_true + y_pred)
    if union == 0.0:
        return 1.0
    iou = (intersec) / (union- intersec)
    return iou

# Function to compute Precision
def precision(y_true, y_pred):
    y_true_f = np.ravel(y_true)
    y_pred_f = np.ravel(y_pred)
    true_positives = np.sum(y_true_f * y_pred_f)
    predicted_positives = np.sum(y_pred_f)
    return (true_positives) / (predicted_positives)

# Function to compute Recall
def recall(y_true, y_pred):
    y_true_f = np.ravel(y_true)
    y_pred_f = np.ravel(y_pred)
    true_positives = np.sum(y_true_f * y_pred_f)
    possible_positives = np.sum(y_true_f)
    return (true_positives) / (possible_positives)

def precision_score_(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    y_true_new = K.flatten(y_true)
    y_pred_new = K.flatten(y_pred)
    intersect = K.sum(y_pred_new*y_true_new)
    total_pixel_pred = K.sum(y_pred_new)
    if total_pixel_pred == 0.0:
        return 1.0
    #precision = intersect/total_pixel_pred
    return intersect/total_pixel_pred

def recall_score_(y_true, y_pred):
    #y_pred = tf.where(y_pred < 0.2, tf.zeros_like(y_pred), y_pred)
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    y_true_new = K.flatten(y_true)
    y_pred_new = K.flatten(y_pred)
    intersect = K.sum(y_pred_new*y_true_new)
    total_pixel_truth = K.sum(y_true_new)
    if total_pixel_truth == 0.0:
        return 1.0
    #recall = np.mean(intersect/total_pixel_truth)
    return intersect/total_pixel_truth


# Initialize lists to store loss and metric values
loss_values = []
dice_values = []
iou_values = []
precision_values = []
recall_values = []
hausdorff_distance_value = []

# Loop through the test generator
for batch_x, batch_y in test_generator:
    # Predict the output for the batch
    #print(batch_y.shape)
    mask_image = np.expand_dims(batch_y, axis=-1)
    y_predwts = model.predict(batch_x)
    #y_predwt = y_predwts
    #print('y_pred',y_predwts.shape)
    y_pred_thresholded = np.where(y_predwts >= 0.5, 1.0, 0.0).astype(np.float32)#binary
    
    # Loop through each sample in the batch
    for i in range(len(batch_x)):
        # Compute the loss and metrics for each sample
        #loss = compute_loss(batch_y[i], y_pred[i])
        dice = dice_coeff(batch_y[i], y_pred_thresholded[i])
        iou_value = iou(batch_y[i], y_pred_thresholded[i])
        precision_value = precision_score_(batch_y[i], y_pred_thresholded[i])
        recall_value = recall_score_(batch_y[i], y_pred_thresholded[i])
        Hausdorff_Distance = hausdorff_distance(mask_image[i], y_predwts[i])
        
        # Store the computed values
        #loss_values.append(loss)
        dice_values.append(dice)
        iou_values.append(iou_value)
        precision_values.append(precision_value)
        recall_values.append(recall_value)
        hausdorff_distance_value.append(Hausdorff_Distance)
    
    # Stop if we've processed all steps
    if len(loss_values) >= len(test_generator):
        break

# Compute the average loss and metrics
#average_loss = np.mean(loss_values)
average_dice = np.mean(dice_values)
average_iou = np.mean(iou_values)
average_precision = np.mean(precision_values)
average_recall = np.mean(recall_values)
#average_hausdorff_distance_value = np.mean(hausdorff_distance_value)
# Convert the list to a tensor for easy aggregation
hd_distances = tf.stack(hausdorff_distance_value)
#Exclude slices with inf values from the calculation
valid_hd_distances = tf.boolean_mask(hd_distances, tf.math.is_finite(hd_distances))
mean_hd = tf.reduce_mean(valid_hd_distances)
max_hd = tf.reduce_max(valid_hd_distances)

#print("Average test loss: ", average_loss)
print("Average test dice: ", average_dice)
print("Average test IoU: ", average_iou)
print("Average test precision: ", average_precision)
print("Average test recall: ", average_recall)
print("Average test hausdorff_distance: ", mean_hd)
print("Max test hausdorff_distance: ", max_hd)
